Imporatation des bibliotheques

In [13]:
import os
import re
import tempfile
import shutil
import streamlit as st
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

Charger VENV

In [14]:
load_dotenv(override=True)

True

Initialisation LLM

In [ ]:
# Initialiser le LLM Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv('GOOGLE_API_KEY'),
    temperature=0  # reponses factuelles et reproductibles
)

Chargement du fichier et extraction du texte brute

In [ ]:
def load_cv(path):                      #lire et charger le fichier.
    pdf_loader = PyPDFLoader(path)
    pages = pdf_loader.load()
    return pages

Pretraitement => Nettoyer le texte : espaces, sauts de ligne, caractères parasites

In [ ]:
def clean_text(text):                                               
    text = re.sub(r'\n+', '\n', text)    # remplacer les sauts de ligne multiples par un seul
    text = re.sub(r'\s{2,}', ' ', text)  # remplacer les espaces multiples par un seul
    return text.strip()

Chunking => Decouper le CV en segments exploitables pour la recherche

In [18]:
def chunk_document(text, chunk_size,overlap):
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder( #init le text splitter
        encoding_name='o200k_base',
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )
    chunks = text_splitter.split_documents(text) # create_documents transforme un texte brut en liste de 'Documents' LangChain
    return chunks

Embedding => Transformer les chunks en vecteurs numériques

In [19]:
def build_embeddings(chunks, model):
    # embedding_model = GoogleGenerativeAIEmbeddings ( model=model, google_api_key=os.getenv('GOOGLE_API_KEY'))  ==>  initialisation du modele d'embeddings
    # output_vector = embedding_model.embed_query(chunks)
    return model

chromaDB => Base de donnees vectorielle

In [ ]:
def create_vector_store(chunks, embeddings):     # stocker les vecteurs dans une base vectorielle ChromaDB
    persist_dir = "./chroma_cv_store"

    vectorstore = Chroma.from_documents(     # ChromaDB calcule automatiquement l'embedding de chaque chunk
        documents=chunks,
        embedding=embeddings,
        collection_name="cv_active",
        persist_directory=persist_dir
    )
    
    return vectorstore

Recherche => Récupérer les k chunks les plus proches de la question

In [21]:
def retrieve_relevant_chunks(question,store, k):
    retriever = store.as_retriever(     # creer le retriever de ChromaDB qui va faire la recherche de similarite
        search_type='similarity',
        search_kwargs={'k': {k}}
    )
    
    docs_recuperes = retriever.invoke(question)
    return docs_recuperes

Construire le prompt

In [22]:
def build_prompt(question, context):
    prompt_template = """
    Tu es l'assistant virtuel personnel de l'étudiant à qui appartient ce CV.
    Réponds à la question du recruteur EN TE BASANT UNIQUEMENT sur le contexte fourni issu de son CV.
    
    Règles de sécurité absolues :
    - Si la réponse ne figure pas dans le CV (le contexte ci-dessous), tu DOIS répondre : "Je ne peux pas répondre à cette question car l'information ne se trouve pas dans mon CV."
    - Ne réponds JAMAIS à partir de tes connaissances générales.
    - Sois professionnel, courtois et précis.
    - Ne donne que des informations véridiques issues du texte.

    <context>
    {context}
    </context>

    <question>
    {question}
    </question>
    """

    return  prompt_template.format(context=context, question=question)

Prompt => Construire l'instruction envoyée au modèle

In [23]:
def generate_answer(prompt):
    response = llm.invoke(prompt)
    return response.content

Sécurité métier => Refuser les questions hors CV

In [24]:
def answer_cv_only(question, store):
    chunks = retrieve_relevant_chunks(question, store, k=4)
    context = "\n\n".join([doc.page_content for doc in chunks])
    prompt = build_prompt(question, context)
    answer = generate_answer(prompt)
    return answer

Chargement dynamique => Permettre l'upload d'un nouveau CV et recharger automatiquement la base vectorielle

In [ ]:
def reload_vector_store(file): 
    
    file_path = "./" + file.name                                # 1. Creer un chemin physique simple dans le dossier courant
    with open(file_path, "wb") as f:                            # 2. Sauvegarder le contenu du fichier uploade sur le disque
        f.write(file.getbuffer())
        
    raw_text = load_cv(file_path)                               # 3. executer le pipeline d'ingestion sur ce fichier
    cleaned_text = clean_text(raw_text)
    chunks = chunk_document(cleaned_text, chunk_size=300, overlap=20)
    embeddings = build_embeddings(chunks, build_embeddings())
    store = create_vector_store(chunks, embeddings)
    
    os.remove(file_path)                                         # 4. supprimer le fichier physique maintenant qu'il est dans la base vectorielle
    
    return store